# EXP-001 — Dummy baseline

## 01 — Thông tin
Hồ sơ theo kế hoạch chung EXP-001–020. Phân tích kết quả chạy lại đã lưu; ngày đồng bộ 19/11/2026 không phải ngày huấn luyện. Mã nguồn cũ `THAI-EXP-001` chỉ dùng truy vết.

In [1]:
from pathlib import Path
import json
import hashlib
import yaml
import pandas as pd
from IPython.display import display, Image

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/wesad/models.py").is_file())
CONFIG_PATH = ROOT / 'experiments/01_baselines/EXP-001_Dummy/runs/20261114/config.yaml'
CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
RESULTS_PATH = ROOT / 'experiments/01_baselines/EXP-001_Dummy/runs/20261114/results.csv'
FOLDER = CONFIG_PATH.parent
PROVENANCE = json.loads((FOLDER / "provenance.json").read_text(encoding="utf-8"))

## 02 — Câu hỏi và giả thuyết
Đối chứng dự đoán lớp phổ biến đạt điểm nào? Dùng để kiểm tra lợi ích của mô hình học từ đặc trưng.
Kết quả có thể tăng hoặc giảm; không điều chỉnh thiết kế theo outer test.

## 03 — Dữ liệu
Bảng physiological68, giữ nguyên schema, checksum, nhãn và 15 subject theo cấu hình đã chạy. Không đưa metadata vào X.

In [2]:
feature_path = ROOT / CONFIG["features"]
if feature_path.is_file():
    assert hashlib.sha256(feature_path.read_bytes()).hexdigest() == CONFIG["feature_sha256"]
for name, expected in PROVENANCE["source_hashes"].items():
    assert hashlib.sha256((ROOT / name).read_bytes()).hexdigest() == expected
assert hashlib.sha256(RESULTS_PATH.read_bytes()).hexdigest() == PROVENANCE["source_hashes"][CONFIG["source_archive"] + "/results.csv"]
print("Đã đối chiếu nguồn và checksum; không huấn luyện lại.")

Đã đối chiếu nguồn và checksum; không huấn luyện lại.


## 04 — Protocol
LOSO 15 người; binary và multiclass tách riêng. Imputer/scaler học trong train. Không tuning trong thí nghiệm này.

## 05 — Cấu hình
`config.yaml` là cấu hình đọc hồ sơ theo mã chung. `training_settings` và `source_config` giữ cấu hình lịch sử với mã nguồn cũ; không sửa manifest để giả như đã train bằng mã mới.

In [3]:
display(CONFIG)

{'experiment_id': 'EXP-001', 'record_kind': 'analysis_of_saved_rerun', 'legacy_run_id': 'THAI-EXP-001', 'source_archive': 'experiments/01_baselines/THAI-EXP-001_Dummy/r02', 'source_config': 'experiments/01_baselines/THAI-EXP-001_Dummy/r02/config.yaml', 'source_notebook': 'notebooks/experiments/Thai_EXP-001_working_R02.ipynb', 'features': 'data/features/g2_physiological68_group15_20261107/all_subjects.csv', 'schema': 'data/features/g2_physiological68_group15_20261107/feature_schema.json', 'feature_sha256': '2f2fddd678f333a800f2869d2e78b407934b2c3492190ed70e28d8a9107fd8ba', 'tasks': ['multiclass', 'binary'], 'seed': 42, 'reference_experiment_id': None, 'training_settings': {'protocol': 'notebook_research_new_runs_v1', 'feature_version': 'physiological_68_v1', 'feature_sha256': '2f2fddd678f333a800f2869d2e78b407934b2c3492190ed70e28d8a9107fd8ba', 'source_paths': ['src/wesad/data.py', 'src/wesad/evaluation.py', 'src/wesad/models.py', 'src/wesad/utils.py'], 'source_hash_normalization': 'LF', 

## 06 — Nguồn thực thi
Notebook này đọc kết quả đã lưu, mặc định không train. Code chạy thật và cấu hình nằm tại `notebooks/experiments/Thai_EXP-001_working_R02.ipynb` và `experiments/01_baselines/THAI-EXP-001_Dummy/r02`. Notebook nguồn giữ code và output của lần chạy cũ; dùng runner chung ở `src/wesad/` cho các baseline. Không đổi tên API/module G3/G4 chỉ để đổi nhãn giai đoạn.

## 07 — Kết quả
Mean và SD (ddof=1) tính theo 15 người; không dùng điểm gộp cửa sổ thay cho mean theo subject.

In [4]:
metrics = pd.read_csv(RESULTS_PATH)
assert len(metrics) == 30 and not metrics.duplicated(["task", "subject_id"]).any()
assert metrics.groupby("task").size().to_dict() == {"binary": 15, "multiclass": 15}
display(metrics)
summary = metrics.groupby("task")[["macro_f1", "balanced_accuracy"]].agg(["mean", "std"])
display(summary)

   subject_id        task  model  n_test  macro_f1  balanced_accuracy  seconds
0         S10  multiclass  dummy     426  0.231084           0.333333   0.2125
1         S11  multiclass  dummy     417  0.234990           0.333333   0.0927
2         S13  multiclass  dummy     417  0.234318           0.333333   0.0986
3         S14  multiclass  dummy     410  0.232114           0.333333   0.0790
4         S15  multiclass  dummy     418  0.233954           0.333333   0.0832
5         S16  multiclass  dummy     415  0.235049           0.333333   0.0983
6         S17  multiclass  dummy     426  0.231084           0.333333   0.0924
7          S2  multiclass  dummy     395  0.237785           0.333333   0.0743
8          S3  multiclass  dummy     401  0.234787           0.333333   0.1121
9          S4  multiclass  dummy     404  0.236422           0.333333   0.0790
10         S5  multiclass  dummy     414  0.238095           0.333333   0.0566
11         S6  multiclass  dummy     412  0.236155  

## 08 — Hình minh họa
Hình dựng từ kết quả/dự đoán đã lưu, dùng mã EXP chung. Không huấn luyện lại. Ảnh mã cũ trong thư mục chỉ dùng truy vết.

In [5]:
image_paths = sorted((FOLDER / "figures").glob(CONFIG["experiment_id"] + "_*.png"))
assert len(image_paths) == 5
for image_path in image_paths:
    display(Image(filename=str(image_path)))

<IPython.core.display.Image object>
<IPython.core.display.Image object>
<IPython.core.display.Image object>
<IPython.core.display.Image object>
<IPython.core.display.Image object>


## 09 — Phân tích và đối chứng
Dummy là mức đối chứng; điểm gần nhau không chứng minh mô hình hữu ích.

In [6]:
comparison_path = ROOT / CONFIG["source_archive"] / "comparison_by_subject.csv"
if comparison_path.is_file():
    comparison = pd.read_csv(comparison_path)
    display(comparison)
    delta_columns = [c for c in comparison if c.endswith("_delta")]
    if delta_columns:
        display(comparison.groupby("task")[delta_columns].agg(["mean", "std"]))
for task, part in metrics.groupby("task"):
    display(part.nsmallest(3, "macro_f1")[["task", "subject_id", "macro_f1", "balanced_accuracy"]])

      task subject_id  macro_f1  balanced_accuracy
15  binary        S10  0.405858                0.5
21  binary        S17  0.405858                0.5
18  binary        S14  0.409222                0.5
         task subject_id  macro_f1  balanced_accuracy
0  multiclass        S10  0.231084           0.333333
6  multiclass        S17  0.231084           0.333333
3  multiclass        S14  0.232114           0.333333


## 10 — Kết luận và nguồn gốc
- binary: Macro-F1 0.412281 ± 0.003526; Balanced Accuracy 0.500000 ± 0.000000.
- multiclass: Macro-F1 0.235047 ± 0.002305; Balanced Accuracy 0.333333 ± 0.000000.

Kết quả biến động giữa người; không kết luận mô hình cuối chỉ từ một hướng. Chọn đặc trưng là nghiên cứu mở rộng sau khi đã quan sát test. Kết quả không phải kiểm định trên tập độc lập mới.

Nguồn cấu hình, checksum và commit được ghi tại `provenance.json`. So sánh train–test của EXP-001/002/011/016 ở `notebooks/04_evaluation/EXP_LR_train_test_comparison.ipynb`.